# Baseline Ladder: Financial Sentiment Adaptation

Compare **adaptation methods** and **backbones** on the same FinancialPhraseBank split.

See **`ORG_CHARTER.md`** for the organizational question and success metrics.

| Experiment | Backbone | Method |
|------------|----------|--------|
| `distilbert_eval_only` | DistilBERT | No training (fresh head) |
| `distilbert_head_only` | DistilBERT | Train classifier only |
| `distilbert_lora` | DistilBERT | LoRA (r=8) |
| `distilbert_dora` | DistilBERT | DoRA |
| `distilbert_pissa` | DistilBERT | PiSSA init |
| `finbert_eval_only` | FinBERT | Pretrained (off-the-shelf) |
| `finbert_head_only` | FinBERT | Train classifier only |
| `finbert_lora` / `dora` / `pissa` | FinBERT | PEFT variants |

Results are saved to `experiments/results/baseline_ladder_results.csv`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "finetuning_lib").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from experiments.run_baseline_ladder import LADDER_EXPERIMENTS, RESULTS_DIR
from finetuning_lib.data import load_financial_phrasebank, make_stratified_splits, remap_dataset_labels
from finetuning_lib.train_utils import MODEL_CONFIGS, get_device, run_experiment, tokenize_datasets
from transformers import AutoTokenizer

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {get_device()}")

## Configuration

Set `QUICK_RUN = True` for a smoke test (1 epoch, seed 42, DistilBERT experiments only).

In [ ]:
QUICK_RUN = True  # False for full ladder (3 epochs, 3 seeds, all experiments)

NUM_EPOCHS = 1 if QUICK_RUN else 3
SEEDS = [42] if QUICK_RUN else [42, 43, 44]

if QUICK_RUN:
    EXPERIMENTS_TO_RUN = [
        e for e in LADDER_EXPERIMENTS
        if e[1] == "distilbert" and e[2] in ("eval_only", "head_only", "lora", "dora")
    ]
else:
    EXPERIMENTS_TO_RUN = LADDER_EXPERIMENTS

print(f"Epochs: {NUM_EPOCHS} | Seeds: {SEEDS} | Experiments: {len(EXPERIMENTS_TO_RUN)}")

## Load data (same splits as main notebook)

In [ ]:
df = load_financial_phrasebank(agreement="all").to_pandas()
dataset = make_stratified_splits(df, random_state=42)
print(dataset)
print(df["label"].value_counts())

## Run ladder

Alternatively from a terminal:

```bash
python experiments/run_baseline_ladder.py --quick
python experiments/run_baseline_ladder.py
```

In [ ]:
device = get_device()
tokenized_cache = {}
tokenizer_cache = {}
all_rows = []

for seed in SEEDS:
    for exp_id, backbone, method in EXPERIMENTS_TO_RUN:
        cfg = MODEL_CONFIGS[backbone]
        if backbone not in tokenized_cache:
            tokenizer_cache[backbone] = AutoTokenizer.from_pretrained(cfg.model_id)
            backbone_data = dataset
            if cfg.label2id != MODEL_CONFIGS["distilbert"].label2id:
                backbone_data = remap_dataset_labels(dataset, cfg.label2id)
            tokenized_cache[backbone] = tokenize_datasets(
                backbone_data, tokenizer_cache[backbone]
            )

        out_dir = str(RESULTS_DIR / "checkpoints" / f"{exp_id}_seed{seed}")
        print(f"\n{'=' * 60}\n{exp_id} | seed={seed}\n{'=' * 60}")

        row = run_experiment(
            experiment_id=exp_id,
            backbone=backbone,
            method=method,
            tokenized=tokenized_cache[backbone],
            tokenizer=tokenizer_cache[backbone],
            device=device,
            output_dir=out_dir,
            num_train_epochs=NUM_EPOCHS,
            seed=seed,
        )
        all_rows.append(row)

results = pd.DataFrame(all_rows)
results.to_csv(RESULTS_DIR / "baseline_ladder_results.csv", index=False)
display(results.sort_values("test_f1_macro", ascending=False))

## Summary statistics

In [ ]:
summary = (
    results.groupby("experiment_id")["test_f1_macro"]
    .agg(["mean", "std", "min", "max", "count"])
    .sort_values("mean", ascending=False)
)
summary.to_csv(RESULTS_DIR / "baseline_ladder_summary.csv")
display(summary)

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(summary))))
summary["mean"].plot(kind="barh", ax=ax, color="steelblue", xerr=summary["std"].fillna(0))
ax.set_xlabel("Test F1-macro (mean ± std over seeds)")
ax.set_title("Baseline ladder — FinancialPhraseBank (all-agree)")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_ladder_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved chart: {RESULTS_DIR / 'baseline_ladder_chart.png'}")

## Interpretation guide

- **`distilbert_eval_only`**: Pretrained encoder + **untrained** head — same role as the main notebook baseline.
- **`finbert_eval_only`**: **Domain-pretrained** model — answers "how far does finance pretraining alone get us?"
- **`head_only`**: Is LoRA necessary, or is a linear head enough?
- **`lora` / `dora` / `pissa`**: PEFT variant comparison at fixed rank; tune learning rate before over-interpreting small gaps (see *Learning Rate Matters*, 2026).

Next steps for the organization: slice eval, multi-seed gates in `ORG_CHARTER.md` Phase 2, and internal label sets.